In [ ]:
import pandas as pd

# ---- PATH TO CLEANED PIREPS ----
PIREPS_PATH = "clean_pirep_data/2024/02_turb_pireps.csv"   # change if needed

# Load dataframe
df = pd.read_csv(
    PIREPS_PATH,
    parse_dates=["datetime"]
)

print("Total cleaned PIREPs:", len(df))

df.head()


In [ ]:
# Pick a LOW-ALTITUDE PIREP where we know reflectivity data exists,
# but force 10 candidates to demonstrate tournament-style selection.
# At FL090 (9000ft), the closest 5 radars have decent coverage. By pulling
# 10 we can see how the tournament scores and ranks them.
pirep = df.iloc[20]  # C208 at FL090, MOD turbulence, Northern CA

print(pirep)
print(f"\nFlight Level: {pirep['FL']:.0f} ft ({pirep['FL']/3.281:.0f} m)")

## Step 1: Scale Turbulence Intensity

Turbulence intensity is adjusted based on aircraft weight category:
- **Light (L)**: no change
- **Medium (M)**: +1
- **Heavy (H)**: +2
- **Unknown (U)**: no change

This normalizes reports since heavier aircraft feel less turbulence in the same conditions.

In [ ]:
import sys
sys.path.insert(0, "..")

from plane_weights.scale_turbulence import scale_turbulence

raw_turb = pirep['turbulence_intensity']
weight = pirep['Plane Weight']

scaled_turb = scale_turbulence(raw_turb, weight)

print(f"Aircraft: {pirep['AIRCRAFT']}")
print(f"Plane Weight Category: {weight}")
print(f"Raw Turbulence Intensity: {raw_turb}")
print(f"Scaled Turbulence Intensity: {scaled_turb}")

## Step 2: Find Candidate NEXRAD Radar Sites (Altitude-Aware)

For high-altitude PIREPs, the 5 closest radars may not have beam coverage at the flight level. We use `beam_geometry` to:
1. Pull more candidates (5/10/20 depending on altitude)
2. Score each by how many elevation angles cover the PIREP altitude band
3. Rank by a combined coverage + distance score

In [ ]:
import numpy as np
from scipy.spatial import cKDTree
from haversine import haversine

sys.path.append("../radars")
from beam_geometry import score_radar_for_pirep, get_num_candidates

# Load NEXRAD site locations
nexrad_sites = pd.read_csv("../radars/nexrad_sites.csv")
nexrad_coords = nexrad_sites[['Latitude', 'Longitude']].to_numpy()

# Build KD-tree in radians (approximates great-circle distance)
nexrad_tree = cKDTree(np.radians(nexrad_coords))
site_codes = nexrad_sites['Site Code'].to_numpy()
site_elevations = nexrad_sites['Elevation'].to_numpy() / 3.281  # ft -> meters

def ft_to_meters(ft):
    return ft / 3.281

altitude_ft = pirep['FL']
altitude_m = ft_to_meters(altitude_ft)

# Force 10 candidates to demonstrate tournament (normally 5 at this altitude)
num_candidates = 10
normal_num = get_num_candidates(altitude_ft)

print(f"PIREP Location: ({pirep['LAT']:.4f}, {pirep['LON']:.4f})")
print(f"PIREP Altitude: {altitude_ft:.0f} ft ({altitude_m:.0f} m)")
print(f"Normal candidates for this altitude: {normal_num}")
print(f"Using {num_candidates} candidates to demonstrate tournament")
print()

# Query KDTree for N nearest stations
pirep_coord = np.radians([pirep['LAT'], pirep['LON']])
_distances, indices = nexrad_tree.query(pirep_coord, k=num_candidates)

# Score each candidate by beam geometry
scored = []
for idx in indices:
    site_lat, site_lon = nexrad_coords[idx]
    distance_m = haversine(
        (pirep['LAT'], pirep['LON']),
        (site_lat, site_lon),
        unit='m'
    )
    score = score_radar_for_pirep(distance_m, altitude_m, site_elevations[idx])
    scored.append((score, site_codes[idx], distance_m / 1000))

scored.sort(key=lambda x: x[0], reverse=True)

print(f"{'Rank':<5} {'Site':<6} {'Score':<8} {'Distance (km)':<15}")
print("-" * 35)
for i, (score, code, dist_km) in enumerate(scored):
    marker = " <-- top 5" if i < 5 else ""
    print(f"{i+1:<5} {code:<6} {score:<8.3f} {dist_km:<15.1f}{marker}")

closest_sites = tuple(code for _score, code, _dist in scored)

## Step 3: Query S3 for Available Radar Scans

For each candidate site, query the `unidata-nexrad-level2` S3 bucket to find all available radar scan times around the PIREP datetime.

In [ ]:
import boto3
from botocore import UNSIGNED
from botocore.config import Config
from datetime import datetime, timedelta
import bisect

NEXRAD_BUCKET = 'unidata-nexrad-level2'  # Migrated from noaa-nexrad-level2 (deprecated Sep 2025)

s3 = boto3.client('s3', region_name='us-east-1', config=Config(signature_version=UNSIGNED))

pirep_dt = pd.to_datetime(pirep['datetime'])
print(f"PIREP datetime: {pirep_dt}")
print()


def get_file_time(filename, date):
    """Extract datetime from a NEXRAD filename."""
    filetime = filename.split("_")[1]
    if ".tar" in filename or "MDM" in filename or filetime == "NEXRAD":
        return None
    hour, minute, second = int(filetime[:2]), int(filetime[2:4]), int(filetime[4:6])
    return datetime(year=date.year, month=date.month, day=date.day,
                    hour=hour, minute=minute, second=second)


def list_nexrad_files(date, site):
    """List all NEXRAD files for a given date and site from S3."""
    prefix = f"{date.year}/{date.month:02}/{date.day:02}/{site}"
    response = s3.list_objects_v2(Bucket=NEXRAD_BUCKET, Prefix=prefix)
    files = response.get("Contents", [])
    filetimes = []
    for f in files:
        basename = f['Key'].rsplit("/", 1)[-1]
        if "_MDM" in basename:
            basename = basename[:basename.index("_MDM")]
        dt = get_file_time(basename, date)
        if dt is not None:
            filetimes.append((dt, basename))
    return filetimes


# For each site, gather available scan times (current day + neighbors if near midnight)
site_scan_times = {}
days_to_check = {pirep_dt.date()}

# Add neighboring days if PIREP is within 30 min of midnight
if (pirep_dt - timedelta(minutes=30)).day != pirep_dt.day:
    days_to_check.add((pirep_dt - timedelta(minutes=30)).date())
if (pirep_dt + timedelta(minutes=30)).day != pirep_dt.day:
    days_to_check.add((pirep_dt + timedelta(minutes=30)).date())

print(f"Days to check: {sorted(days_to_check)}")
print()

for site in closest_sites:
    times = []
    for day in sorted(days_to_check):
        times += list_nexrad_files(day, site)
    site_scan_times[site] = sorted(times, key=lambda x: x[0])
    print(f"{site}: {len(times)} scans available")

## Step 4: Select the Closest Radar Scan (Most Recent Before PIREP)

For each site, find the radar scan most recently *before* the PIREP time. We only use past scans — no future data leakage.

In [ ]:
aws_files = []

for site in closest_sites:
    times = site_scan_times[site]
    if len(times) == 0:
        print(f"  {site}: No scans available, skipping")
        continue
    
    # Find the most recent scan before the PIREP time
    scan_datetimes = [t[0] for t in times]
    idx = bisect.bisect_left(scan_datetimes, pirep_dt)
    
    # Get the scan just before the PIREP
    if idx > 0:
        closest_dt, file_ending = times[idx - 1]
    else:
        closest_dt, file_ending = times[0]
    
    delta = pirep_dt - closest_dt
    prefix = f"{closest_dt.year}/{closest_dt.month:02}/{closest_dt.day:02}/{file_ending[:4]}"
    s3_path = f"s3://{NEXRAD_BUCKET}/{prefix}/{file_ending}"
    aws_files.append(s3_path)
    
    print(f"  {site}: {file_ending} (Δt = {delta})")

print(f"\nSelected {len(aws_files)} radar files")
print(f"\nClosest radar file (will be used for gridding):")
print(f"  {aws_files[0]}")

## Step 5: Tournament-Style Radar Selection

Download all candidate radar files, then:
1. **Phase 1**: Grid the 5 closest together. If NaN fraction ≤ 90%, use them as-is (fast path).
2. **Phase 2**: If coverage is poor, score each radar individually, drop useless ones (≥99% NaN), and tournament the remaining candidates to fill back up to 5.

In [ ]:
import pyart
import tempfile
import os
from create_grid import create_grid

# Grid parameters (must match the pipeline)
NUM_Z_POINTS = 10
NUM_Y_POINTS = 16
NUM_X_POINTS = 16
grid_shape = (NUM_Z_POINTS, NUM_Y_POINTS, NUM_X_POINTS)

DEGREES = 0.25
Z_SIZE = 3048  # meters (10,000 ft)
alt_limits = (-Z_SIZE / 2.0, Z_SIZE / 2.0)
lat_limits = (-DEGREES / 2.0, DEGREES / 2.0)
lon_limits = (-DEGREES / 2.0, DEGREES / 2.0)

pirep_alt_m = ft_to_meters(pirep['FL'])
grid_origin = (pirep_alt_m, pirep['LAT'], pirep['LON'])

MAX_RADARS_IN_GRID = 5
NAN_THRESHOLD = 0.90
INDIVIDUAL_NAN_CUTOFF = 0.99

def download_radar(s3_path):
    """Download a NEXRAD file from S3 and return a pyart Radar object, or None."""
    key = s3_path.replace(f"s3://{NEXRAD_BUCKET}/", "")
    try:
        with tempfile.NamedTemporaryFile(delete=False) as tmp:
            tmp_path = tmp.name
        s3.download_file(NEXRAD_BUCKET, key, tmp_path)
        radar = pyart.io.read_nexrad_archive(tmp_path)
        os.unlink(tmp_path)
        
        # Fix longitude calibration issue
        if radar.longitude['data'][0] == 0:
            site_code = key.split("/")[3]
            site_lon = nexrad_sites.loc[nexrad_sites['Site Code'] == site_code, 'Longitude'].iloc[0]
            radar.gate_longitude['data'] += site_lon
            radar.longitude['data'][0] = site_lon
        return radar
    except Exception as e:
        if os.path.exists(tmp_path):
            os.unlink(tmp_path)
        print(f"  Failed to load {s3_path.split('/')[-1]}: {e}")
        return None

def score_single_radar(radar):
    """Grid a single radar and return its NaN fraction."""
    trial_grid = create_grid(
        radars=radar, grid_shape=grid_shape,
        alt_range=alt_limits, lat_range=lat_limits, lon_range=lon_limits,
        grid_origin=grid_origin, fields=["reflectivity"],
        map_roi=False, verbose=False
    )
    if not trial_grid:
        return 1.0
    return trial_grid.attrs.get("nan_fraction", 1.0)

print(f"Grid origin: alt={pirep_alt_m:.0f}m ({pirep['FL']:.0f}ft), "
      f"lat={pirep['LAT']:.4f}°, lon={pirep['LON']:.4f}°")
print(f"Total candidate radar files: {len(aws_files)}")
print()

# --- Phase 1: Try the closest 5 ---
first_five = aws_files[:MAX_RADARS_IN_GRID]
extra_candidates = aws_files[MAX_RADARS_IN_GRID:]

print("=== Phase 1: Loading closest 5 radars ===")
initial_radars = []
for rf in first_five:
    print(f"  Downloading {rf.split('/')[-1]}...", end=" ")
    radar = download_radar(rf)
    if radar is not None:
        initial_radars.append((rf, radar))
        print("OK")

if len(initial_radars) > 0 and len(extra_candidates) > 0:
    # Grid the initial 5 together to check coverage
    initial_tuple = tuple(r for _rf, r in initial_radars)
    initial_grid = create_grid(
        radars=initial_tuple, grid_shape=grid_shape,
        alt_range=alt_limits, lat_range=lat_limits, lon_range=lon_limits,
        grid_origin=grid_origin, fields=["reflectivity"],
        map_roi=False, verbose=False
    )
    initial_nan_frac = 1.0 if not initial_grid else initial_grid.attrs.get("nan_fraction", 1.0)
    print(f"\nInitial 5 combined NaN fraction: {initial_nan_frac:.3f}")
    
    if initial_nan_frac <= NAN_THRESHOLD:
        print(f"Coverage is acceptable (≤ {NAN_THRESHOLD}), skipping tournament")
        selected_radars = list(initial_tuple)
    else:
        # --- Phase 2: Tournament ---
        print(f"Coverage is poor (> {NAN_THRESHOLD}), starting tournament...")
        print(f"\nScoring initial 5 individually:")
        kept = []
        for rf, radar in initial_radars:
            nan_frac = score_single_radar(radar)
            name = rf.split('/')[-1]
            if nan_frac < INDIVIDUAL_NAN_CUTOFF:
                kept.append((nan_frac, radar))
                print(f"  KEEP  {name}: nan_fraction={nan_frac:.3f}")
            else:
                print(f"  DROP  {name}: nan_fraction={nan_frac:.3f}")
        
        slots_to_fill = MAX_RADARS_IN_GRID - len(kept)
        print(f"\nKept {len(kept)}/5, need {slots_to_fill} replacement(s)")
        
        if slots_to_fill > 0:
            print(f"\nTournament: scoring {len(extra_candidates)} extra candidates...")
            extras_scored = []
            for rf in extra_candidates:
                name = rf.split('/')[-1]
                print(f"  Downloading {name}...", end=" ")
                radar = download_radar(rf)
                if radar is None:
                    continue
                nan_frac = score_single_radar(radar)
                if nan_frac < INDIVIDUAL_NAN_CUTOFF:
                    extras_scored.append((nan_frac, radar))
                    print(f"nan_fraction={nan_frac:.3f}")
                else:
                    print(f"nan_fraction={nan_frac:.3f} (dropped)")
            
            extras_scored.sort(key=lambda x: x[0])
            kept.extend(extras_scored[:slots_to_fill])
        
        kept.sort(key=lambda x: x[0])
        selected_radars = [r for _nf, r in kept[:MAX_RADARS_IN_GRID]]
else:
    selected_radars = [r for _rf, r in initial_radars]

print(f"\n=== Final: {len(selected_radars)} radars selected for combined grid ===")

## Step 6: Create the Final Combined 3D Grid

Grid the selected radars together into a **10 x 16 x 16** (altitude x latitude x longitude) grid centered on the PIREP location.

In [ ]:
print(f"Creating final combined grid with {len(selected_radars)} radars...")
print(f"Grid shape: {grid_shape}")
print(f"Grid volume: {DEGREES}° x {DEGREES}° x {Z_SIZE}m")
print()

if len(selected_radars) == 0:
    print("WARNING: No radars with any coverage — cannot create grid.")
    grid = None
else:
    grid = create_grid(
        radars=tuple(selected_radars),
        grid_shape=grid_shape,
        alt_range=alt_limits,
        lat_range=lat_limits,
        lon_range=lon_limits,
        grid_origin=grid_origin,
        fields=["reflectivity"],
        map_roi=False,
        verbose=True
    )

    if not grid:
        print("\nWARNING: No radar data found within the grid volume!")
    else:
        nan_frac = grid.attrs.get("nan_fraction", -1)
        print(f"\nGrid created successfully!")
        print(f"NaN fraction: {nan_frac:.3f}")
        print(f"Radars used: {grid.attrs.get('num_radars', '?')}")
        print(grid)

## Step 7: Visualize the Grid

Plot a slice of the reflectivity grid to see what the model input looks like.

In [ ]:
import matplotlib.pyplot as plt

if grid:
    reflectivity = grid['reflectivity'].values  # shape: (10, 16, 16)
    
    # Count non-NaN cells
    valid_cells = np.count_nonzero(~np.isnan(reflectivity))
    total_cells = reflectivity.size
    print(f"Valid (non-NaN) cells: {valid_cells}/{total_cells} ({100*valid_cells/total_cells:.1f}%)")
    
    # Plot the middle altitude slice
    mid_alt_idx = NUM_Z_POINTS // 2
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Left: middle altitude slice
    im = axes[0].imshow(
        reflectivity[mid_alt_idx, :, :],
        origin='lower',
        extent=[grid.lon.values.min(), grid.lon.values.max(),
                grid.lat.values.min(), grid.lat.values.max()],
        cmap='NWSRef',
        vmin=-10, vmax=60
    )
    axes[0].set_xlabel('Longitude (°)')
    axes[0].set_ylabel('Latitude (°)')
    axes[0].set_title(f'Reflectivity at Alt Index {mid_alt_idx} '
                      f'({grid.alt.values[mid_alt_idx]:.0f}m)')
    axes[0].plot(pirep['LON'], pirep['LAT'], 'k*', markersize=15, label='PIREP')
    axes[0].legend()
    plt.colorbar(im, ax=axes[0], label='Reflectivity (dBZ)')
    
    # Right: vertical cross-section at middle latitude
    mid_lat_idx = NUM_Y_POINTS // 2
    im2 = axes[1].imshow(
        reflectivity[:, mid_lat_idx, :],
        origin='lower',
        extent=[grid.lon.values.min(), grid.lon.values.max(),
                grid.alt.values.min(), grid.alt.values.max()],
        aspect='auto',
        cmap='NWSRef',
        vmin=-10, vmax=60
    )
    axes[1].set_xlabel('Longitude (°)')
    axes[1].set_ylabel('Altitude (m)')
    axes[1].set_title(f'Vertical Cross-Section at Lat Index {mid_lat_idx}')
    axes[1].axhline(y=pirep_alt_m, color='black', linestyle='--', label='PIREP altitude')
    axes[1].legend()
    plt.colorbar(im2, ax=axes[1], label='Reflectivity (dBZ)')
    
    plt.tight_layout()
    plt.show()
else:
    print("No grid data to visualize.")

## Step 8: Package as Model Input (NetCDF)

Attach PIREP metadata as attributes and save the grid as a NetCDF file — the final model input format.

In [ ]:
if grid:
    # Compute time delta between PIREP and closest radar scan
    radar_file = aws_files[0]
    key_parts = radar_file.replace(f"s3://{NEXRAD_BUCKET}/", "").split("/")
    radar_date = datetime(year=int(key_parts[0]), month=int(key_parts[1]), day=int(key_parts[2]))
    radar_basename = key_parts[-1]
    radar_t = get_file_time(radar_basename, radar_date)
    delta_t = (pirep_dt - radar_t).seconds

    # Attach metadata as attributes
    grid.attrs = {
        "LAT": pirep["LAT"],
        "LON": pirep["LON"],
        "ALT": pirep["FL"],
        "DELTA_T": delta_t,
        "TURB": scaled_turb,
        "NUM_RADARS": len(selected_radars),
        "NAN_FRACTION": float(grid.attrs.get("nan_fraction", -1.0)),
    }

    print("Model Input Attributes:")
    for k, v in grid.attrs.items():
        print(f"  {k}: {v}")

    # Save to NetCDF
    output_path = "single_pirep_model_input.nc"
    grid.to_netcdf(output_path)
    print(f"\nSaved model input to: {output_path}")
else:
    print("No grid data — cannot create model input.")

## Step 9: Convert to Model Feature Vector

This is what the PyTorch `RadarDataLoader` does — flatten the 3D grid into a 1D feature vector, prepend the metadata, and replace NaN values with -32.0.

In [ ]:
import torch
import xarray as xr

if grid:
    # Re-read the saved NetCDF (simulating what the dataloader does)
    ds = xr.open_dataset("single_pirep_model_input.nc")
    
    # This mirrors dataloader_class.py exactly:
    # 1. Extract all attributes as an array (order: LAT, LON, ALT, DELTA_T, TURB)
    attrs_arr = np.array(list(ds.attrs.values()))
    
    # 2. Features = all attributes except the last (TURB), which becomes the label
    features = attrs_arr[:-1].astype(float)
    
    # 3. Flatten all data variables (reflectivity grid: 10x16x16 -> 2560)
    flattened_data = np.concatenate([ds[var].values.flatten() for var in ds.data_vars])
    
    # 4. Concatenate metadata + flattened grid
    features = np.concatenate((features, flattened_data))
    
    # 5. Label is the last attribute (TURB), cast to int
    label = int(attrs_arr[-1])
    
    # 6. Convert to tensor and replace NaN with -32.0
    feature_tensor = torch.tensor(features, dtype=torch.float32)
    feature_tensor = torch.nan_to_num(feature_tensor, nan=-32.0)
    
    print(f"Feature vector shape: {feature_tensor.shape}")
    print(f"  - Metadata (4): LAT={attrs_arr[0]:.4f}, LON={attrs_arr[1]:.4f}, "
          f"ALT={attrs_arr[2]:.0f}, DELTA_T={attrs_arr[3]:.0f}s")
    print(f"  - Reflectivity ({flattened_data.size}): flattened 10x16x16 grid")
    print(f"  - NaN cells replaced with -32.0")
    print(f"Label (scaled turbulence): {label}")
    print()
    print(f"This (feature_tensor, label) pair is one training sample for the model.")
    
    ds.close()
else:
    print("No grid data available.")

## Step 10: NaN Fraction Survey Across Altitudes

Sample 5 PIREPs per altitude band (50 total) with turbulence (intensity ≥ 1) to understand the NaN distribution. For each, we grid the 5 closest radars and report the NaN fraction.

In [ ]:
import importlib
import create_grid as cg
importlib.reload(cg)
from create_grid import create_grid

# Filter to PIREPs with actual turbulence reports
turb_df = df[df['turbulence_intensity'] >= 1].copy()

SAMPLES_PER_BAND = 5

altitude_bands = [
    ("FL030-050",  3000,  5000),
    ("FL050-080",  5000,  8000),
    ("FL080-100",  8000, 10000),
    ("FL100-140", 10000, 14000),
    ("FL140-180", 14000, 18000),
    ("FL180-220", 18000, 22000),
    ("FL220-280", 22000, 28000),
    ("FL280-330", 28000, 33000),
    ("FL330-380", 33000, 38000),
    ("FL380-450", 38000, 45000),
]

test_pireps = []
for label, fl_min, fl_max in altitude_bands:
    band = turb_df[(turb_df['FL'] >= fl_min) & (turb_df['FL'] < fl_max)]
    if len(band) == 0:
        print(f"No turbulence PIREPs in {label}")
        continue
    # Sample evenly spaced PIREPs from the band
    step = max(len(band) // (SAMPLES_PER_BAND + 1), 1)
    indices = [band.index[min(step * (i + 1), len(band) - 1)] for i in range(SAMPLES_PER_BAND)]
    # Deduplicate in case band is small
    indices = list(dict.fromkeys(indices))
    for idx in indices:
        test_pireps.append((label, idx))

print(f"Selected {len(test_pireps)} PIREPs across {len(altitude_bands)} altitude bands")
for label, idx in test_pireps:
    p = df.loc[idx]
    print(f"  {label}: idx={idx:>5}, FL={p['FL']:>6.0f}, turb={p['turbulence_intensity']:.0f}, "
          f"({p['LAT']:>7.2f}, {p['LON']:>8.2f})")

In [ ]:
def process_pirep_for_survey(pirep_row, num_radars=5):
    """
    Run the full pipeline for a single PIREP and return the NaN fraction.
    Downloads num_radars closest radar files, grids them, returns stats.
    """
    p = df.loc[pirep_row]
    p_dt = pd.to_datetime(p['datetime'])
    p_alt_m = ft_to_meters(p['FL'])
    p_origin = (p_alt_m, p['LAT'], p['LON'])
    
    # Find closest radar sites
    p_coord = np.radians([p['LAT'], p['LON']])
    _d, inds = nexrad_tree.query(p_coord, k=num_radars)
    sites = site_codes[inds]
    
    # Get scan times for each site
    days = {p_dt.date()}
    if (p_dt - timedelta(minutes=30)).day != p_dt.day:
        days.add((p_dt - timedelta(minutes=30)).date())
    if (p_dt + timedelta(minutes=30)).day != p_dt.day:
        days.add((p_dt + timedelta(minutes=30)).date())
    
    # Find closest scan file for each site
    files = []
    for site in sites:
        times = []
        for day in sorted(days):
            times += list_nexrad_files(day, site)
        times.sort(key=lambda x: x[0])
        if len(times) == 0:
            continue
        scan_dts = [t[0] for t in times]
        idx = bisect.bisect_left(scan_dts, p_dt)
        if idx > 0:
            closest_dt, fname = times[idx - 1]
        else:
            closest_dt, fname = times[0]
        prefix = f"{closest_dt.year}/{closest_dt.month:02}/{closest_dt.day:02}/{fname[:4]}"
        files.append(f"s3://{NEXRAD_BUCKET}/{prefix}/{fname}")
    
    if len(files) == 0:
        return {"nan_fraction": 1.0, "valid_cells": 0, "radars_loaded": 0}
    
    # Download and grid
    radars = []
    for f in files:
        r = download_radar(f)
        if r is not None:
            radars.append(r)
    
    if len(radars) == 0:
        return {"nan_fraction": 1.0, "valid_cells": 0, "radars_loaded": 0}
    
    g = create_grid(
        radars=tuple(radars), grid_shape=grid_shape,
        alt_range=alt_limits, lat_range=lat_limits, lon_range=lon_limits,
        grid_origin=p_origin, fields=["reflectivity"],
        map_roi=False, verbose=False
    )
    
    if not g:
        return {"nan_fraction": 1.0, "valid_cells": 0, "radars_loaded": len(radars)}
    
    refl = g['reflectivity'].values
    valid = int(np.count_nonzero(~np.isnan(refl)))
    nf = g.attrs.get("nan_fraction", 1.0)
    return {"nan_fraction": nf, "valid_cells": valid, "radars_loaded": len(radars)}

# Run the survey
print(f"Processing {len(test_pireps)} PIREPs (5 radars each)...")
print(f"{'Band':<12} {'FL':>6} {'NaN%':>7} {'Valid':>7} {'Radars':>7} {'Status':<10}")
print("-" * 55)

results = []
for i, (label, idx) in enumerate(test_pireps):
    p = df.loc[idx]
    print(f"{label:<12} {p['FL']:>6.0f}", end="", flush=True)
    
    res = process_pirep_for_survey(idx, num_radars=5)
    nan_pct = res['nan_fraction'] * 100
    status = "OK" if nan_pct <= 90 else "SPARSE" if nan_pct < 100 else "EMPTY"
    
    print(f" {nan_pct:>6.1f}% {res['valid_cells']:>7} {res['radars_loaded']:>7} {status:<10}  [{i+1}/{len(test_pireps)}]")
    results.append({"band": label, "FL": p['FL'], **res})

print(f"\nDone! Processed {len(results)} PIREPs.")

In [ ]:
# Aggregate results by band
band_stats = {}
for r in results:
    b = r['band']
    if b not in band_stats:
        band_stats[b] = {'nan_fracs': [], 'valid_cells': [], 'FL': r['FL']}
    band_stats[b]['nan_fracs'].append(r['nan_fraction'])
    band_stats[b]['valid_cells'].append(r['valid_cells'])

bands_ordered = [label for label, _, _ in altitude_bands if label in band_stats]

# Box plot of NaN fraction per band
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

# Top: box plot
box_data = [np.array(band_stats[b]['nan_fracs']) * 100 for b in bands_ordered]
bp = ax1.boxplot(box_data, patch_artist=True, labels=bands_ordered)
for patch, median_line in zip(bp['boxes'], bp['medians']):
    median_val = median_line.get_ydata()[0]
    if median_val <= 90:
        patch.set_facecolor('lightgreen')
    elif median_val < 100:
        patch.set_facecolor('lightyellow')
    else:
        patch.set_facecolor('lightsalmon')
ax1.axhline(y=90, color='red', linestyle='--', alpha=0.7, label='90% filter threshold')
ax1.set_ylabel('NaN Fraction (%)')
ax1.set_title(f'NaN Fraction Distribution by Altitude Band ({SAMPLES_PER_BAND} samples each, 5 radars)')
ax1.legend()
ax1.set_ylim(0, 105)
ax1.tick_params(axis='x', rotation=30)

# Bottom: scatter plot of all individual results
for i, r in enumerate(results):
    color = 'green' if r['nan_fraction'] <= 0.90 else 'orange' if r['nan_fraction'] < 1.0 else 'red'
    ax2.scatter(r['FL'] / 1000, r['nan_fraction'] * 100, c=color, s=60, edgecolors='black', alpha=0.7)
ax2.axhline(y=90, color='red', linestyle='--', alpha=0.7, label='90% filter threshold')
ax2.set_xlabel('Flight Level (thousands of feet)')
ax2.set_ylabel('NaN Fraction (%)')
ax2.set_title('NaN Fraction vs Flight Level (all individual samples)')
ax2.legend()
ax2.set_ylim(0, 105)

plt.tight_layout()
plt.show()

# Summary table
print(f"\n{'Band':<12} {'Count':>6} {'Mean NaN%':>10} {'Min NaN%':>10} {'Max NaN%':>10} {'≤90% NaN':>10}")
print("-" * 62)
total_ok = 0
total_n = 0
for b in bands_ordered:
    nfs = np.array(band_stats[b]['nan_fracs']) * 100
    ok_count = int(np.sum(nfs <= 90))
    total_ok += ok_count
    total_n += len(nfs)
    print(f"{b:<12} {len(nfs):>6} {np.mean(nfs):>9.1f}% {np.min(nfs):>9.1f}% {np.max(nfs):>9.1f}% {ok_count:>5}/{len(nfs)}")
print("-" * 62)
print(f"{'TOTAL':<12} {total_n:>6} {'':>10} {'':>10} {'':>10} {total_ok:>5}/{total_n}")
print(f"\nOverall pass rate (≤90% NaN): {total_ok}/{total_n} ({100*total_ok/total_n:.1f}%)")

In [ ]:
# Save survey results to CSV
results_df = pd.DataFrame(results)
results_df['nan_pct'] = results_df['nan_fraction'] * 100
results_df['status'] = results_df['nan_fraction'].apply(
    lambda x: 'OK' if x <= 0.90 else 'SPARSE' if x < 1.0 else 'EMPTY'
)
results_df.to_csv("nan_fraction_survey_results.csv", index=False)
print(f"Saved survey results to nan_fraction_survey_results.csv")
print(results_df.to_string(index=False))